In [1]:
import os
# 禁止 JAX 预占所有显存，用多少拿多少
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from pennylane.fermi import FermiSentence, from_string
import pennylane as qml
from pennylane import numpy as np
import time
qml.about()
import catalyst

Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_catalyst, pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)


In [2]:
import jax
print('✅ JAX version:', jax.devices())
import jax
print('✅ JAX version:', jax.__version__)
print('✅ Devices:', jax.devices())
if any(d.platform == 'gpu' for d in jax.devices()):
    print('🎉 GPU is working!')
else:
    print('⚠️ No GPU detected')

✅ JAX version: [CudaDevice(id=0)]
✅ JAX version: 0.6.2
✅ Devices: [CudaDevice(id=0)]
🎉 GPU is working!


In [3]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=6 N=4.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_3 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_3.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_3.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_3.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_3.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=3, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

✅ 矩阵调用成功！
矩阵格式：csr
矩阵形状：(11424, 11424)
非零元素个数：548306
最小特征值: [-28.08061951 -11.19353886   8.38332622]
-28.08061951456625


In [4]:
import pennylane as qml
import jax
import jax.numpy as jnp
import numpy as np  # 用于预处理，非计算图部分
import time
import optax  # JAX 的标准优化库 (需要 pip install optax)

from catalyst import qjit, grad

# =================== 0. 辅助函数 (保持不变) ===================
def get_Hami(H):
    # 安全地将 H 转换为无梯度的 NumPy 数组
    if hasattr(H, 'toarray'):
        H = H.toarray()
    if hasattr(H, 'detach'):
        H = H.detach().cpu().numpy()
    elif hasattr(H, 'numpy'):
        H = H.numpy()
    else:
        H = np.asarray(H)

    H_dense = np.array(H, copy=True)
    d = H_dense.shape[0]
    Nq = int(np.ceil(np.log2(d)))
    l = 2 ** Nq

    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    np.fill_diagonal(Hami, 1000)
    Hami[:d, :d] = H_dense

    return Hami, Nq

H_sy,Nq = get_Hami(H)



In [5]:
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np

H_sy,n_qubits = get_Hami(H)
H_sy = coo_matrix(H_sy)
# ---------- 1. 生成 Gray 码 ----------
def gray_code(n: int) -> list[str]:
    """返回 n-bit Gray 码列表，顺序对应 0..2^n-1"""
    if n == 1:
        return ["0", "1"]
    lower = gray_code(n - 1)
    return ["0" + x for x in lower] + ["1" + x for x in reversed(lower)]

gray_basis   = gray_code(Nq)                        # len == 2**Nq
gray2natural = np.array([int(g, 2) for g in gray_basis], dtype=np.int64)

# ---------- 2. 构造 Gray-ordered 哈密顿量 ----------

rows, cols = H_sy.nonzero()          # 默认是 int32/uint32
data       = H_sy.data

# 关键修复：先把索引数组变成 int64，再做高级索引
rows = rows.astype(np.int64, copy=False)
cols = cols.astype(np.int64, copy=False)

new_rows = gray2natural[rows]
new_cols = gray2natural[cols]

print('data  len:', len(data))
print('rows  len:', len(new_rows))
print('cols  len:', len(new_cols))

# 现在三者长度一致，且索引不会溢出
H_gray = coo_matrix((data, (new_rows, new_cols)),
                    shape=(2**Nq, 2**Nq)).tocsr()

H_gray = H_gray.toarray()








data  len: 553266
rows  len: 553266
cols  len: 553266


In [7]:
from jax import numpy as jnp
import scipy.sparse as sp
depth = 100


seed = 42
H = jnp.array(H_gray, dtype=jnp.complex128)
H_hermitian = qml.Hermitian(H, wires=range(n_qubits))

hf = np.zeros(n_qubits)
# 使用 lightning.gpu 设备
dev = qml.device("lightning.gpu", wires=n_qubits)

@qml.qjit
@qml.qnode(dev)
def cost(params):

    qml.BasisState(hf, wires=range(n_qubits))
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    for d in range(depth):
        for i in range(n_qubits):
            qml.RY(params[d*n_qubits+i], wires=i)

        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

    return qml.expval(H_hermitian)


n = depth * n_qubits
init_params = jnp.zeros(n)

cost(init_params)



/home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages/jax/_src/interpreters/mlir.py:1135: UserWarning: A large amount of constants were captured during lowering (4.29GB total). If this is intentional, disable this warning by setting JAX_CAPTURED_CONSTANTS_WARN_BYTES=-1. To obtain a report of where these constants were encountered, set JAX_CAPTURED_CONSTANTS_REPORT_FRAMES=-1.
  warnings.warn(message)


OSError: [Errno 122] Disk quota exceeded

In [7]:

import catalyst
import optax
steps = 5000
lr = 0.001
#opt = optax.adam(0.001)


opt = optax.adabelief(learning_rate=0.01)

opt_state = opt.init(init_params)

params = init_params


@qml.qjit
def update_step(i, params, opt_state):
    """Perform a single gradient update step"""
    energy, grads = catalyst.value_and_grad(cost)(params)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    # catalyst.debug.print("Step = {i},  Energy = {energy:.8f} Ha", i=i, energy=energy)
    return (params, opt_state, energy)



for i in range(steps):
    params, opt_state,energy = update_step(i, params, opt_state)
    if i % 50 == 0:
        print(f"Step = {i},  Energy = {energy:.8f} Ha")

# -8.48179737

Step = 0,  Energy = 410.18707119 Ha
Step = 50,  Energy = 7.17876742 Ha
Step = 100,  Energy = -0.50227570 Ha
Step = 150,  Energy = -4.06243836 Ha
Step = 200,  Energy = -6.25841932 Ha
Step = 250,  Energy = -7.33349629 Ha
Step = 300,  Energy = -7.70641251 Ha
Step = 350,  Energy = -7.97744584 Ha
Step = 400,  Energy = -8.12212011 Ha
Step = 450,  Energy = -8.21493901 Ha
Step = 500,  Energy = -8.28447605 Ha
Step = 550,  Energy = -8.34087066 Ha
Step = 600,  Energy = -8.29567533 Ha
Step = 650,  Energy = -8.40574872 Ha
Step = 700,  Energy = -8.06082064 Ha
Step = 750,  Energy = -8.43431291 Ha
Step = 800,  Energy = -8.45238872 Ha
Step = 850,  Energy = -8.31463299 Ha
Step = 900,  Energy = -8.46570936 Ha
Step = 950,  Energy = -8.46976617 Ha
Step = 1000,  Energy = -8.39307297 Ha
Step = 1050,  Energy = -8.47493392 Ha
Step = 1100,  Energy = -8.47647117 Ha
Step = 1150,  Energy = -8.41386507 Ha
Step = 1200,  Energy = -7.83531751 Ha
Step = 1250,  Energy = -8.47027713 Ha
Step = 1300,  Energy = -8.44142132 

In [8]:
import concurrent.futures
import copy

def run_optimizer(name, opt, init_params, steps):
    # 注意：必须在函数内部定义 qjit，避免跨线程共享编译对象
    @qml.qjit
    def update_step(i, params, opt_state):
        energy, grads = catalyst.value_and_grad(cost)(params)
        updates, opt_state = opt.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, energy

    params = copy.deepcopy(init_params)
    opt_state = opt.init(params)
    history = []

    for i in range(steps):
        params, opt_state, energy = update_step(i, params, opt_state)
        if i % 50 == 0:
            print(f"[{name}] Step = {i}, Energy = {energy:.8f} Ha")
            history.append((i, float(energy)))
    return name, history

# 主程序
optimizers = {
    "adam": optax.adam(0.001),
    #"rmsprop": optax.rmsprop(0.02),
    #"sgd": optax.sgd(0.05),
}

with concurrent.futures.ThreadPoolExecutor(max_workers=len(optimizers)) as executor:
    futures = [
        executor.submit(run_optimizer, name, opt, init_params, steps)
        for name, opt in optimizers.items()
    ]
    results = dict(f.result() for f in concurrent.futures.as_completed(futures))

[adam] Step = 0, Energy = 410.18707119 Ha
[adam] Step = 50, Energy = 17.31503280 Ha
[adam] Step = 100, Energy = 11.51048743 Ha
[adam] Step = 150, Energy = 7.32346615 Ha
[adam] Step = 200, Energy = 3.73907802 Ha
[adam] Step = 250, Energy = 1.25071317 Ha
[adam] Step = 300, Energy = -0.73757584 Ha
[adam] Step = 350, Energy = -2.32236317 Ha
[adam] Step = 400, Energy = -3.55995806 Ha
[adam] Step = 450, Energy = -4.49955302 Ha
[adam] Step = 500, Energy = -5.20396480 Ha
[adam] Step = 550, Energy = -5.75936887 Ha
[adam] Step = 600, Energy = -6.22500311 Ha
[adam] Step = 650, Energy = -6.62758768 Ha
[adam] Step = 700, Energy = -6.97755619 Ha
[adam] Step = 750, Energy = -7.28165747 Ha
[adam] Step = 800, Energy = -7.54291557 Ha
[adam] Step = 850, Energy = -7.76036703 Ha
[adam] Step = 900, Energy = -7.93364050 Ha
[adam] Step = 950, Energy = -8.06650302 Ha
[adam] Step = 1000, Energy = -8.16604737 Ha
[adam] Step = 1050, Energy = -8.24000060 Ha
[adam] Step = 1100, Energy = -8.29491931 Ha
[adam] Step =

In [9]:
@qml.qjit
def optimization(params):
    opt_state = opt.init(params)
    (params, opt_state) = qml.for_loop(0, steps, 1)(update_step)(params, opt_state)
    return params
final_params = optimization(init_params)

# print(f"Final angle parameters: {final_params}")

ValueError: too many values to unpack (expected 2)

In [ ]:


# =================== 1. 准备数据 ===================


# 获取处理后的 Hamiltonian
Hami_np, n_qubits = get_Hami(H)

# !!! 关键点：将 NumPy 数组转换为 JAX 数组，并移动到 GPU (如果可用) !!!
Hami = jnp.array(Hami_np)
print(f"Qubits: {n_qubits}")

# =================== 2. 超参数与设备 ===================
depth = 100
steps = 2000 # JAX 很快，可以跑更多步，或者演示用少一点
lr = 0.05
seed = 42

# 使用 lightning.gpu 设备
dev = qml.device("lightning.gpu", wires=n_qubits)

# =================== 3. 定义电路 (QNode) ===================
# 注意：我们这里不直接装饰 @qjit，而是稍后在优化步中一起编译
@qml.qnode(dev)
def circuit_node(params, hamiltonian_matrix):
    params = params.reshape((depth, n_qubits))

    for d in range(depth):
        for i in range(n_qubits):
            qml.RY(params[d, i], wires=i)

        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

    # 使用 Hermitian 算符
    return qml.expval(qml.Hermitian(hamiltonian_matrix, wires=range(n_qubits)))

# =================== 4. 定义优化步骤 (JIT 编译核心) ===================

# 初始化优化器 (Optax)
optimizer = optax.adam(learning_rate=lr)

# 定义损失函数
def cost_fn(params, hamiltonian_matrix):
    return circuit_node(params, hamiltonian_matrix)

# 定义单步更新函数，并加上 @qjit 编译整个流程
# 这会将 梯度计算 + 参数更新 + 电路执行 全部编译成机器码
@qjit
def update_step(params, opt_state, hamiltonian_matrix):
    # 1. 计算损失和梯度
    loss, grads = jax.value_and_grad(cost_fn)(params, hamiltonian_matrix)

    # 2. 计算参数更新量
    updates, new_opt_state = optimizer.update(grads, opt_state)

    # 3. 应用更新
    new_params = optax.apply_updates(params, updates)

    return new_params, new_opt_state, loss

# =================== 5. 训练循环 ===================
print(f"\n>>> 开始 JAX+Catalyst 优化（共 {steps} 步）")

# 初始化参数 (使用 JAX 的随机键)
key = jax.random.PRNGKey(seed)
init_params = jax.random.uniform(key, shape=(depth * n_qubits,), minval=0, maxval=2*jnp.pi)

# 初始化优化器状态
opt_state = optimizer.init(init_params)

params = init_params

# 计时开始
# 注意：第一次运行包含编译时间 (Compile time)
tic = time.time()

for step in range(steps + 1):
    params, opt_state, loss = update_step(params, opt_state, Hami)

    # 将 loss 转回 CPU 打印 (这会触发同步，不要每一步都打印，否则会拖慢速度)
    if step % 50 == 0:
        # float(loss) 会强制同步数据
        print(f"  step {step:3d}  energy = {float(loss):.8f}")

toc = time.time()

print(f"Final Energy = {loss:.8f}")
print(f"Total Time   = {toc-tic:.2f}s (含编译时间)")

In [ ]:
import pennylane as qml
import time
from pennylane import numpy as np

# ===================  超参数  ===================
n_qubits = 9
depth    = 100
steps    = 10000
seed     = 42
np.random.seed(seed)



# ===================  设备  ===================
dev = qml.device("lightning.gpu", wires=n_qubits)

# ===================  Ansatz  ===================
def ansatz(params):
    params = params.reshape((depth, n_qubits))
    for d in range(depth):
        for i in range(n_qubits):

            qml.RY(params[d, i], wires=i)

        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev)
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)) )

# ===================  优化器列表  ===================
optimizers = {
    "Adam"     : (qml.optimize.AdamOptimizer, 0.05, {}),

}

# ===================  训练循环  ===================
results = {}
for name, (OptClass, lr, kw) in optimizers.items():
    print(f"\n>>> 开始 {name} 优化（共 {steps} 步）")
    params = np.random.uniform(0, 2*np.pi, (depth,  n_qubits), requires_grad=True)
    opt    = OptClass(lr, **kw) if lr is not None else OptClass(**kw)

    tic = time.time()
    for step in range(steps + 1):
        params, loss = opt.step_and_cost(energy_fn, params)
        if step % 50 == 0:
            print(f"  step {step:3d}  energy = {loss:.8f}")
    toc = time.time()

    final_e = energy_fn(params)
    results[name] = final_e
    print(f"{name:10s} 最终能量 = {final_e:.8f}  耗时 = {toc-tic:.2f}s")

# ===================  汇总  ===================
print("\n" + "="*50)
for name, e in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:10s}  {e:.8f}")


In [ ]:
import pennylane as qml
import time
from pennylane import numpy as np
from scipy.optimize import minimize

# ===================  超参数  ===================
n_qubits = 9
depth    = 30
steps    = 1000          # 仅用于 PennyLane 优化器
seed     = 42
np.random.seed(seed)

# ===================  设备  ===================
dev = qml.device("lightning.gpu", wires=n_qubits)   # 没 GPU 就换 lightning.qubit

# ===================  哈密顿量  ===================


# ===================  Ansatz  ===================
def ansatz(params):
    params = params.reshape((depth, n_qubits))
    for d in range(depth):
        for i in range(n_qubits):
            qml.RY(params[d, i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev)
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

# ---------- 扁平接口，供 scipy 使用 ----------
def flat_energy(p_flat):
    return float(energy_fn(p_flat))

def flat_grad(p_flat):
    return qml.grad(energy_fn)(p_flat)

# ===================  优化器列表  ===================
# 对 PennyLane 优化器用 step_and_cost；对 SLSQP 用 minimize
optimizers = {
    #"Adam" : (qml.optimize.AdamOptimizer, 0.05, {}),
    #"SPSA" : (qml.optimize.SPSAOptimizer, 0.02, {"c": 0.3}),
    "SLSQP": None,   # 特殊处理
}

# ===================  训练循环  ===================
results = {}
for name in optimizers:
    print(f"\n>>> 开始 {name} 优化")
    x0 = np.zeros(depth*n_qubits, requires_grad=True)

    tic = time.time()
    if name == "SLSQP":
        res = minimize(flat_energy, x0=x0, method="SLSQP",
                       options={"maxiter": 1000, "ftol": 1e-8})
        final_e, params = res.fun, res.x
    else:
        OptClass, lr, kw = optimizers[name]
        opt = OptClass(lr, **kw)
        for step in range(steps + 1):
            x0, loss = opt.step_and_cost(energy_fn, x0)
            if step % 100 == 0:
                print(f"  step {step:4d}  energy = {loss:.8f}")
        final_e = energy_fn(x0)
        params = x0
    toc = time.time()

    results[name] = final_e
    print(f"{name:10s} 最终能量 = {final_e:.8f}  耗时 = {toc-tic:.2f}s")

# ===================  汇总  ===================
print("\n" + "="*50)
for name, e in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:10s}  {e:.8f}")


In [ ]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer,RotosolveOptimizer

##############################
# 1. 问题参数：比特数、HEA 深度
##############################
n_qubits = 9
N = n_qubits             # 任意 N 比特
depth = 20         # HEA 层数
steps = 400        # VQE 迭代步数
lr = 0.05          # Adam 学习率

##############################
# 2. 定义哈密顿量（Transverse-Field Ising）
##############################
         # hx

##############################
# 3. 构造 Hardware-Efficient Ansatz
#    每层：单比特旋转 RX-RY-RX + CNOT 纠缠
##############################
def hea_layer(params):
    """单个 HEA 层：params 形状 (3, N)"""
    for i in range(N):
        qml.RX(params[0, i], wires=i)
        qml.RY(params[1, i], wires=i)
        qml.RX(params[2, i], wires=i)
    # 线性邻接 CNOT 纠缠
    for i in range(N - 1):
        qml.CNOT(wires=[i, i + 1])

def ansatz(params):
    """depth 层 HEA"""
    params = params.reshape((depth, 3, N))
    for d in range(depth):
        hea_layer(params[d])

##############################
# 4. 定义 QNode：返回能量期望值
##############################
dev = qml.device("lightning.gpu", wires=N)

@qml.qnode(dev, interface="autograd")
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

##############################
# 5. 初始化参数并运行 VQE
##############################
params = np.zeros((depth, 3, N), requires_grad=True)

opt = qml.optimize.NesterovMomentumOptimizer(0.01, 0.9)

print("开始 VQE 优化...")
for step in range(steps + 1):
    params, loss = opt.step_and_cost(energy_fn, params)
    if step % 10 == 0:
        print(f"Step {step:3d} | energy = {loss:.8f}")

print("\n最终基态能量估计:", energy_fn(params))

##############################
# 6. 可选：打印最终线路
##############################
# 在 PennyLane 0.43 里，只要这样即可：
# print(qml.draw(energy_fn)(params))




In [ ]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer, RotosolveOptimizer

##############################
# 1. 问题参数：比特数、HEA 深度
##############################
n_qubits = 12
N = n_qubits             # 任意 N 比特
depth = 20               # HEA 层数
steps = 400              # VQE 迭代步数
lr = 0.05                # Adam 学习率（Rotosolve 不用）

##############################
# 2. 定义哈密顿量（Transverse-Field Ising）
##############################
# 这里你原来怎么定义 Hami 就怎么来
# Hami = ...

##############################
# 3. 构造 Hardware-Efficient Ansatz
##############################
def hea_layer(params):
    """单个 HEA 层：params 形状 (3, N)"""
    for i in range(N):
        qml.RX(params[0, i], wires=i)
        qml.RY(params[1, i], wires=i)
        qml.RX(params[2, i], wires=i)
    # 线性邻接 CNOT 纠缠
    for i in range(N - 1):
        qml.CNOT(wires=[i, i + 1])

def ansatz(params):
    """depth 层 HEA"""
    params = params.reshape((depth, 3, N))
    for d in range(depth):
        hea_layer(params[d])

##############################
# 4. 定义 QNode：返回能量期望值
##############################
dev = qml.device("lightning.gpu", wires=N)

@qml.qnode(dev, interface="autograd")
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

##############################
# 5. 初始化参数
##############################
params = np.random.uniform(0, 2 * np.pi, size=(depth, 3, N), requires_grad=True)

##############################
# 5.1 为 Rotosolve 设置频率信息（关键！！）
##############################
# params 的名字就是 energy_fn 的参数名 "params"
# 形状 (depth, 3, N)，每个标量参数都只在一个 RX/RY/RX 旋转里，
# 对应的频率个数 = 1
nums_frequency = {
    "params": { (d, k, i): 1
                for d in range(depth)
                for k in range(3)
                for i in range(N) }
}

##############################
# 5.2 选择 Rotosolve 优化器
##############################
opt = qml.RotosolveOptimizer()

print("开始 VQE 优化（Rotosolve）...")
for step in range(steps + 1):
    params, loss = opt.step_and_cost(
        energy_fn,
        params,
        nums_frequency=nums_frequency,   # ⭐ 一定要传这一项
    )
    if step % 10 == 0:
        print(f"Step {step:3d} | energy = {loss:.8f}")

print("\n最终基态能量估计:", energy_fn(params))

##############################
# 6. 可选：打印最终线路
##############################
# print(qml.draw(energy_fn)(params))
